# Overlap gap: validation report vs. platform

**Question.** The validation report (`ab_validation_data.xlsx`, sheet
`8_quality_checks`) flags submission **795099854** as overlapping
**796376139** by ~88%. On the platform these two plots do **not** overlap.
Why the difference?

**Answer (TL;DR).** They are measuring **two different geometries**:

| | Geometry used | Field | Overlap 795099854 ↔ 796376139 |
|---|---|---|---|
| Validation `step4` | boundary **as captured** in the field | `Validate Boundary` / `validate_polygon` | **88% / 65%** |
| Platform map & export (`step6`) | boundary **after adjustment** | `Entire Area Polygon` / `Open_Area_GeoMapping` | **0% / 0%** |

The plot was redrawn after capture. `step4` checks the raw captured boundary;
the platform shows the adjusted one. This notebook proves it by recomputing the
overlap directly from both geometry sources.

## Data sources

- **`ab_validation_data.xlsx` → `8_quality_checks`** — `ab_polygon_checks.py`
  output. `step4_*` = *as captured*, `step6_*` = *final boundary now in Kobo*.
- **`ab_platform_data.xlsx` → `Plot Table`** — what the platform stores. The
  `Entire Area Polygon` column is a KML URL of the **final/adjusted** boundary
  (the one drawn on the platform map).
- **`2026-08-14_ab_realdata.csv`** — raw Kobo export; `Validate Boundary` holds
  the **as-captured** vertices (`lat lon alt accuracy`, points separated by `;`).

In [ ]:
import re, ssl, math, urllib.request
import pandas as pd
from shapely.geometry import Polygon
from shapely.ops import transform

REPORTS = "data/reports"
DATA = "data"
VAL_XLSX = f"{REPORTS}/ab_validation_data.xlsx"
PLAT_XLSX = f"{REPORTS}/ab_platform_data.xlsx"
RAW_CSV = f"{DATA}/2026-08-14_ab_realdata.csv"

PAIR = ("795099854", "796376139")  # the two submissions in question
pd.set_option("display.max_columns", None, "display.width", 200)

In [ ]:
def captured_poly(s):
    """As-captured boundary from the raw CSV: 'lat lon alt acc' points, ';'-sep."""
    pts = []
    for tok in re.split(r"[;\n]", str(s)):
        p = tok.split()
        if len(p) >= 2:
            lat, lon = float(p[0]), float(p[1])
            pts.append((lon, lat))
    return Polygon(pts).buffer(0)  # buffer(0) fixes self-touching rings


_KML = {}  # tiny cache so re-runs do not re-fetch


def kml_poly(url):
    """Final/adjusted boundary from the platform KML endpoint ('lon,lat,alt')."""
    if url not in _KML:
        req = urllib.request.Request(url, headers={"User-Agent": "curl/8"})
        ctx = ssl.create_default_context()
        xml = urllib.request.urlopen(req, timeout=30, context=ctx).read().decode()
        _KML[url] = xml
    coords = re.search(r"<coordinates>(.*?)</coordinates>", _KML[url], re.S).group(1)
    pts = []
    for tok in coords.split():
        c = tok.split(",")
        if len(c) >= 2:
            pts.append((float(c[0]), float(c[1])))  # lon, lat
    return Polygon(pts).buffer(0)


def _to_m(poly, lat0):
    """Local metric projection around lat0 (good enough at plot scale)."""
    sx = 111320 * math.cos(math.radians(lat0))
    return transform(lambda x, y, z=None: (x * sx, y * 110540), poly)


def overlap_pct(a, b):
    """Overlap of a and b as % of each polygon's own area."""
    lat0 = a.centroid.y
    am, bm = _to_m(a, lat0), _to_m(b, lat0)
    inter = am.intersection(bm).area
    return inter / am.area * 100, inter / bm.area * 100


def area_ha(poly):
    return _to_m(poly, poly.centroid.y).area / 1e4

## Step 1 — What the validation report itself already says

Note the report has *two* overlap columns for the pair: `step4` (as captured)
is high, `step6` (final) is already **0**. The contradiction with the platform
only appears if you read `step4` and compare it to the platform map.

In [ ]:
val = pd.read_excel(VAL_XLSX, sheet_name="8_quality_checks")
cols = ["submission_id", "step4_area_ha", "step4_max_overlap_pct",
        "step4_overlaps_with", "step6_final_area_ha",
        "step6_final_max_overlap_pct", "step7_kobo_status"]
pair_rows = val[val.submission_id.isin([int(x) for x in PAIR])][cols]
pair_rows

## Step 2 — Recompute the overlap from both geometry sources

Pull the **as-captured** boundary from the raw CSV and the **final** boundary
from the platform KML, then measure the overlap on each. If the gap is really
"captured vs adjusted", captured should reproduce ~88%/65% and platform ~0%.

In [ ]:
raw = pd.read_csv(RAW_CSV, dtype=str, sep=";").set_index("_id")
plat = pd.read_excel(PLAT_XLSX, sheet_name="Plot Table", dtype=str)
plat["sid"] = plat["Submission ID"].str.lstrip("#")
plat = plat.set_index("sid")

a, b = PAIR
cap = {i: captured_poly(raw.loc[i, "Validate Boundary"]) for i in PAIR}
fin = {i: kml_poly(plat.loc[i, "Entire Area Polygon"]) for i in PAIR}

cap_ov = overlap_pct(cap[a], cap[b])
fin_ov = overlap_pct(fin[a], fin[b])

pd.DataFrame({
    "source": ["CAPTURED (step4 / Validate Boundary)",
               "FINAL (step6 / platform Entire Area Polygon)"],
    f"area_ha_{a}": [area_ha(cap[a]), area_ha(fin[a])],
    f"area_ha_{b}": [area_ha(cap[b]), area_ha(fin[b])],
    f"overlap%_{a}": [cap_ov[0], fin_ov[0]],
    f"overlap%_{b}": [cap_ov[1], fin_ov[1]],
}).round(4)

The recomputed numbers match the report (`step4` 0.8846/0.6543, `step6` 0.0)
and the platform areas (~0.167 ha). So the **gap is not a bug** — `step4` and
the platform are looking at different boundaries of the same plots.

## Step 3 — Is this pair special, or a general pattern?

Across *every* pair the report flags on the captured boundary (`step4`), how
many still overlap after adjustment (`step6`)? This is the general "gap" between
the validation report and the platform.

In [ ]:
flagged = val[val["step4_overlaps_with"].notna()].copy()
flagged["resolved_after_adjustment"] = flagged["step6_final_max_overlap_pct"].fillna(0).eq(0)
summary = pd.DataFrame({
    "plots flagged overlapping as captured (step4)": [len(flagged)],
    "no longer overlapping after adjustment (step6)": [int(flagged["resolved_after_adjustment"].sum())],
    "still overlapping after adjustment": [int((~flagged["resolved_after_adjustment"]).sum())],
}).T.rename(columns={0: "count"})
print(summary)
print("\nIs our pair among the 'resolved after adjustment' set?")
print(flagged[flagged.submission_id.isin([int(x) for x in PAIR])]
      [["submission_id", "step4_max_overlap_pct",
        "step6_final_max_overlap_pct", "resolved_after_adjustment"]]
      .to_string(index=False))

## Conclusion

**The gap is a geometry-version mismatch, not a calculation error.**

- The validation report's `step4_max_overlap_pct` (88% / 65%) is computed on the
  **as-captured** boundary (`Validate Boundary` / `validate_polygon`).
- The platform map and `ab_platform_data.xlsx`'s `Entire Area Polygon` show the
  **adjusted/final** boundary (`Open_Area_GeoMapping`), on which the overlap is
  **0%** — exactly what `step6_final_max_overlap_pct` already reports.
- Plots **795099854** and **796376139** were redrawn after capture (area grew
  from ~0.057/0.076 ha to ~0.167/0.163 ha), which removed the overlap.

**How to read the report correctly:** compare the platform against
`step6_*` (final), not `step4_*` (as captured). `step4` answers "what did the
enumerator capture in the field?"; `step6` and the platform answer "what is the
boundary now?".